In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm
import mlflow.tensorflow
import mlflow.pyfunc

from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.statespace.sarimax import SARIMAX
import xgboost as xgb
import lightgbm as lgb
from prophet import Prophet
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

Importing plotly failed. Interactive plots will not work.


In [2]:
mlflow.set_tracking_uri("sqlite:///F:/Abdelrahman1/ME/programming/SIC/Project3.2/mlflow.db")
mlflow.set_experiment("Solar_Forecast_Comparison")

PROCESSED_DIR = r'F:\Abdelrahman1\ME\programming\SIC\Project3.2\data\processed\Forecast'
input_path = os.path.join(PROCESSED_DIR, 'forecast_features_final.csv')

df = pd.read_csv(input_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

2026/10/28 01:12:22 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/28 01:12:23 INFO mlflow.store.db.utils: Updating database tables
2026/10/28 01:12:43 INFO mlflow.tracking.fluent: Experiment with name 'Solar_Forecast_Comparison' does not exist. Creating a new experiment.


In [3]:
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

target = 'actual_power_kw'
features = [c for c in df.columns if c not in ['timestamp', 'actual_power_kw']]

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

results = []

In [4]:
# 1. Ridge (Linear)
with mlflow.start_run(run_name="Ridge_Linear"):
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train_scaled, y_train)
    preds = ridge.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    
    mlflow.log_param("alpha", 1.0)
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2", r2)
    mlflow.sklearn.log_model(ridge, "model")
    results.append({'Model': 'Ridge', 'MAE': mae, 'R2': r2})

2026/10/28 01:12:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [5]:
# 2. XGBoost (Tree)
with mlflow.start_run(run_name="XGBoost_Tree"):
    xgb_model = xgb.XGBRegressor(n_estimators=150, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1)
    xgb_model.fit(X_train, y_train)
    preds = xgb_model.predict(X_test)
    
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    
    mlflow.log_param("n_estimators", 150)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("max_depth", 6)
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2", r2)
    mlflow.xgboost.log_model(xgb_model, "model")
    results.append({'Model': 'XGBoost', 'MAE': mae, 'R2': r2})

2026/10/28 01:13:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [6]:
# 3. LightGBM (Tree)
with mlflow.start_run(run_name="LightGBM_Tree"):
    lgb_model = lgb.LGBMRegressor(n_estimators=150, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1, verbose=-1)
    lgb_model.fit(X_train, y_train)
    preds = lgb_model.predict(X_test)
    
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    
    mlflow.log_param("n_estimators", 150)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("max_depth", 6)
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2", r2)
    mlflow.lightgbm.log_model(lgb_model, "model")
    results.append({'Model': 'LightGBM', 'MAE': mae, 'R2': r2})

2026/10/28 01:13:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [7]:
# 4. MLP (Deep Learning)
with mlflow.start_run(run_name="MLP_Deep"):
    mlp = MLPRegressor(hidden_layer_sizes=(100, 50), activation='relu', solver='adam', max_iter=500, random_state=42)
    mlp.fit(X_train_scaled, y_train)
    preds = mlp.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    
    mlflow.log_param("hidden_layers", "(100, 50)")
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2", r2)
    mlflow.sklearn.log_model(
        mlp, 
        "model", 
        skops_trusted_types=['sklearn.neural_network._stochastic_optimizers.AdamOptimizer']
    )
    results.append({'Model': 'MLP', 'MAE': mae, 'R2': r2})

2026/10/28 01:16:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [8]:
# 5. Prophet
with mlflow.start_run(run_name="Prophet_TS"):
    prophet_train = train_df[['timestamp', 'actual_power_kw']].rename(columns={'timestamp': 'ds', 'actual_power_kw': 'y'})
    prophet_train['ghi'] = train_df['ghi']
    prophet_train['temperature'] = train_df['temperature']
    
    prophet_test = test_df[['timestamp']].rename(columns={'timestamp': 'ds'})
    prophet_test['ghi'] = test_df['ghi']
    prophet_test['temperature'] = test_df['temperature']

    prophet_model = Prophet(daily_seasonality=True, yearly_seasonality=True)
    prophet_model.add_regressor('ghi')
    prophet_model.add_regressor('temperature')
    prophet_model.fit(prophet_train)
    
    preds = prophet_model.predict(prophet_test)['yhat'].values
    preds = np.clip(preds, 0, None)
    
    mae, rmse, r2 = mean_absolute_error(y_test, preds), np.sqrt(mean_squared_error(y_test, preds)), r2_score(y_test, preds)
    mlflow.log_metric("MAE", mae); mlflow.log_metric("RMSE", rmse); mlflow.log_metric("R2", r2)
    
    class ProphetWrapper(mlflow.pyfunc.PythonModel):
        def __init__(self, model): self.model = model
        def predict(self, context, model_input): return self.model.predict(model_input)['yhat'].values

    mlflow.pyfunc.log_model(artifact_path="model", python_model=ProphetWrapper(prophet_model))
    results.append({'Model': 'Prophet', 'MAE': mae, 'R2': r2})

Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.
01:17:08 - cmdstanpy - INFO - Chain [1] start processing
01:17:58 - cmdstanpy - INFO - Chain [1] done processing
c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/10/28 01:18:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/28 01:18:02 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized usin

In [9]:
# 6. LSTM
with mlflow.start_run(run_name="LSTM_Deep_TS"):
    time_steps = 12
    
    X_train_lstm, y_train_lstm = [], []
    for i in range(time_steps, len(X_train_scaled)):
        X_train_lstm.append(X_train_scaled[i-time_steps:i])
        y_train_lstm.append(y_train.iloc[i])

    X_test_lstm = []
    X_test_padded = np.vstack([X_train_scaled[-time_steps:], X_test_scaled])
    for i in range(time_steps, len(X_test_padded)):
        X_test_lstm.append(X_test_padded[i-time_steps:i])

    X_train_lstm, X_test_lstm = np.array(X_train_lstm), np.array(X_test_lstm)
    y_train_lstm = np.array(y_train_lstm)
    y_test_lstm = y_test.iloc[-len(X_test_lstm):].values

    lstm_model = Sequential([
        LSTM(64, return_sequences=False, input_shape=(time_steps, X_train_scaled.shape[1])),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    lstm_model.compile(optimizer='adam', loss='mse')
    lstm_model.fit(X_train_lstm, y_train_lstm, epochs=10, batch_size=32, verbose=0)

    preds = lstm_model.predict(X_test_lstm).flatten()
    mae, rmse, r2 = mean_absolute_error(y_test_lstm, preds), np.sqrt(mean_squared_error(y_test_lstm, preds)), r2_score(y_test_lstm, preds)
    
    mlflow.log_metric("MAE", mae); mlflow.log_metric("RMSE", rmse); mlflow.log_metric("R2", r2)
    
    mlflow.tensorflow.log_model(lstm_model, "model")
    results.append({'Model': 'LSTM', 'MAE': mae, 'R2': r2})

614/614 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step


2026/10/28 01:19:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/28 01:19:44 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


In [10]:
results_df = pd.DataFrame(results).sort_values('MAE')
print(results_df.to_string(index=False))

   Model       MAE       R2
 XGBoost  2.771516 0.989222
LightGBM  2.841807 0.988149
     MLP  3.821124 0.988468
    LSTM  4.153664 0.977009
   Ridge  4.494220 0.984545
 Prophet 20.644303 0.769002


In [15]:
import os
os.makedirs("Forecast", exist_ok=True)
experiment = mlflow.get_experiment_by_name("Solar_Forecast_Comparison")
experiment_id = experiment.experiment_id

runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    filter_string="tags.mlflow.runName = 'XGBoost_Tree'"
)
if not runs.empty:
    best_run_id = runs.iloc[0]['run_id']
else:
    print("not founded")

model_uri = f"runs:/{best_run_id}/model"
loaded_model = mlflow.xgboost.load_model(model_uri)
loaded_model.save_model("Forecast\\forecast_best_model.json")

--------------------------------------

--------------------------------------